In [1]:
from os.path import join, dirname
import os
import sys
import random
from copy import deepcopy
from hydra import initialize, compose
from omegaconf import DictConfig
import numpy as np
import torchshow as ts
import torchvision

import torch
import torch.nn.functional as F
from einops import rearrange

sys.path.append(join(dirname(dirname(os.getcwd()))))
sys.path.append(os.getcwd())

from omniisaacgymenvs.utils.hydra_cfg.hydra_utils import *
from omniisaacgymenvs.utils.hydra_cfg.reformat import omegaconf_to_dict, print_dict
from omniisaacgymenvs.utils.task_util import initialize_task
from omniisaacgymenvs.envs.vec_env_rlgames import VecEnvRLGames

from memoryPerceiver.memory_perceiver.main.configuration import cf
from memoryPerceiver.memory_perceiver.core.model_wrapper import create_memories, reset_memories
import memoryPerceiver.memory_perceiver.isaac.trainer as tr
from memoryPerceiver.memory_perceiver.isaac.plot import Plotter
from memoryPerceiver.memory_perceiver.isaac.agent import ActorCritic, ActorCriticSimple, ActorCriticPerceiver

%load_ext autoreload
%autoreload 2
%env ISAAC_JUPYTER_KERNEL=1

env: ISAAC_JUPYTER_KERNEL=1


/isaac-sim/kit/python/lib/python3.7/site-packages/IPython/extensions/autoreload.py:121: DeprecationWarning: the imp module is deprecated in favour of importlib; see the module's documentation for alternative uses
  from imp import reload


In [2]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
device_cpu = torch.device("cpu")
random.seed(0)
np.random.seed(0)
torch.use_deterministic_algorithms(False)
%env CUBLAS_WORKSPACE_CONFIG=:4096:8

env: CUBLAS_WORKSPACE_CONFIG=:4096:8


In [3]:
"""model = ActorCriticSimple(cf, device)
model_student = ActorCriticSimple(cf, device)

#actions, logs, vals, dist = model.act(torch.randn(8,4, device=device))
#print(actions.size(), logs.size(), vals.size())
"""

'model = ActorCriticSimple(cf, device)\nmodel_student = ActorCriticSimple(cf, device)\n\n#actions, logs, vals, dist = model.act(torch.randn(8,4, device=device))\n#print(actions.size(), logs.size(), vals.size())\n'

In [4]:
model = ActorCriticPerceiver(cf, device)
model_student = ActorCriticPerceiver(cf, device)
"""
numEnvs=1
states = {'observations' : torch.randn(numEnvs,cf.frames,4, device=device),
         'video' : torch.zeros(numEnvs, 3, cf.frames, 256, 456, device=device),
         'audio':torch.zeros(numEnvs, cf.frames*1920, device=device)
        }
model.act(states)
"""

"\nnumEnvs=1\nstates = {'observations' : torch.randn(numEnvs,cf.frames,4, device=device),\n         'video' : torch.zeros(numEnvs, 3, cf.frames, 256, 456, device=device),\n         'audio':torch.zeros(numEnvs, cf.frames*1920, device=device)\n        }\nmodel.act(states)\n"

In [5]:
#@hydra.main(config_name="config", config_path="../cfg")
#@hydra.main(config_name="config", config_path="../../OmniIsaacGymEnvs/omniisaacgymenvs/cfg")
def parse_hydra_configs(cfg: DictConfig):
    cfg_dict = omegaconf_to_dict(cfg)

    render = True
    env = VecEnvRLGames(headless=cfg.headless, sim_device=cfg.device_id, enable_viewport=cfg.enable_viewport)
    
    
    # !!! Load isaac modules after the variables
    from omni.isaac.core.utils.torch.maths import set_seed
    import omni.usd
    from omni.isaac.core.utils.extensions import enable_extension
    enable_extension("omni.isaac.sensor")
    from omni.isaac.core.utils.stage import open_stage, save_stage
    
    plotter = Plotter(titles=['loss_actor', 'loss_critic', 'rewards'])
    history_keys = ['actions', 'values', 'logprobs', 'returns', 'rewards', 'advantages', 'death_index', 'generator']
    history = tr.init_history(history_keys, sim_duration, numEnvs, cf.dim_actions_ppo, device)
    previous_history = deepcopy(history)
    buffer = {'video':[], 'audio':[], 'obs':[], 'rewards':[], 'actions_star':[]}
    memories = create_memories()

    cfg.seed = set_seed(cfg.seed, torch_deterministic=cfg.torch_deterministic)
    cfg_dict['seed'] = cfg.seed  # sets seed. if seed is -1 will pick a random one
    task = initialize_task(cfg_dict, env)
    all_losses_actor, all_losses_critic, all_returns = [], [], []
    optimizer = torch.optim.Adam(model_student.parameters(), lr=learning_rate)
    sim_count, sim_step = -1, 0
    first_run=True
    is_inference=False  # Will switch to True immediately
    
    while env._simulation_app.is_running():
        if env._world.is_playing():
            # if env._world.current_time_step_index == 0 or \
            # env._world.current_time_step_index % sim_duration == 0:
            if env.sim_frame_count % sim_duration == 0:
                
                ### Plots

                if sim_count % 6 == 1 and len(all_losses_actor)>0:
                    plotter.plot_all([(all_losses_actor, 160), (all_losses_critic, 160), all_returns])

                ### Initialization

                sim_count+=1
                env._world.reset(soft=False)
                is_inference = not is_inference
                
                print('$$$$$$$$$$$ CUMUL REWARD $$$$$$$$$$$')
                if len(history['rewards'])>0:
                    print(history['rewards'].sum())
                if is_inference:
                    print('PLAY')
                    if first_run and os.path.exists(path_model):
                        model.load_state_dict(torch.load(path_model))
                        model_student.load_state_dict(torch.load(path_model))
                    else:
                        if sim_count % student_update_step*2 == 0:
                            model.load_state_dict(model_student.state_dict())
                    reset_memories(memories)
                    torch.save(model_student.state_dict(), path_model)
                    model.eval()
                else:
                    print('REPLAY')
                    first_run=False
                    reset_memories(memories)
                    model_student.train() 
                    optimizer.zero_grad()
                    previous_history = deepcopy(history)
                    previous_history['returns'] = tr.calculate_returns(history['rewards'], history['death_index'], gamma=c_gamma)
                    previous_history['advantages'] = tr.calculate_gae(history['rewards'], history['values'], 
                                                        history['death_index'], gamma=c_gamma, gae_lambda=c_lambda)
                    all_returns.append(previous_history['returns'][1, :].mean().item())
                    print(previous_history['returns'][:200, 1])
                    
                torch.manual_seed(sim_count//2)
                torch.cuda.manual_seed(sim_count//2)
                memories = {i:None for i in range(numEnvs)}
                full_video = []
                birth_time = torch.zeros(numEnvs)
                for key in buffer.keys():
                    buffer[key]=[]
                history = tr.init_history(history_keys, sim_duration, numEnvs, cf.dim_actions_ppo, device)
                backward_count, kl_div = 0, 0
                sim_step = 0

            ### Get agent observations 

            all_obs = env._task.get_observations()['articulation_view']
            obs = all_obs['obs_buf'].clone()    # (numEnvs, d)
            video = all_obs['video_buf'].clone()  # (numEnvs, 256, 456, 3)
            #video = torch.zeros(numEnvs, 256, 456, 3, device=device)
            #ts.save(video[0], '/isaac-sim/memoryPerceiver/memory_perceiver/main/test0.jpg')
            full_video.append(video.cpu())
            env._task.calculate_metrics()       
            reward=env._task.rew_buf.clone()    # (numEnvs, 1)
            tr.update_buffers(buffer, {'obs': obs, 'video': video, 'rewards': reward}, frames=cf.frames)
            history['rewards'][sim_step] = reward

            ### Reset memory at death and record 

            reset_indexes = []
            for agent_i in range(numEnvs):
                if env._task.reset_buf[agent_i].item() == 1:
                    reset_indexes.append(agent_i)
                    birth_time[agent_i] = sim_step

                    # Update death dictionnary during inference

                    if is_inference:
                        history['death_index'][sim_step, agent_i]=1
            reset_memories(memories, indexes=reset_indexes)

            ### Agent
            
            if len(buffer['actions_star'])==0 :

                ## Process inputs

                states={}
                mask = tr.create_mask(birth_time, sim_step, frames=cf.frames)
                states['observations'] = tr.buffer_padding(torch.stack(buffer['obs']), [cf.dim_observations],
                                                            numEnvs=numEnvs, frames=cf.frames)
                states['observations'][mask] *= 0
                states['video'] = torch.zeros(numEnvs, cf.frames, 256, 456, 3, device=device)
                # states['video'] = tr.buffer_padding(torch.stack(buffer['video']), [256, 456, 3], numEnvs=numEnvs, frames=cf.frames)
                states['video'][mask] *= 0
                states['video'] = rearrange(states['video'], 'b t h w c -> b c t h w')
                states['audio'] = torch.zeros(numEnvs, cf.frames*1920, device=device)

                ## Inference, save history and memory update

                if is_inference:
                    actions, logprobs, values, old_dist, memories = model.act(states, memories)   
                    history['actions'][sim_step] = actions  
                    history['logprobs'][sim_step] = logprobs 
                    history['values'][sim_step] = values     
                    history['generator'][sim_step] = torch.cuda.get_rng_state().clone()
                else:
                    actions, logprobs, values = [previous_history[key][sim_step] for key in ['actions', 'logprobs', 'values']]
                    torch.cuda.set_rng_state(previous_history['generator'][sim_step])
                                    

                ## Backward pass

                if not is_inference and sim_step>0:

                    returns = previous_history['returns'][sim_step]
                    advantages = previous_history['advantages'][sim_step]
                    new_logprobs, new_values, entropies, new_dist, memories = model_student.evaluate(states, actions, memories)

                    # Loss
                    ratios = torch.exp(new_logprobs - logprobs)
                    surr1 = ratios * advantages
                    surr2 = torch.clamp(ratios, 1-eps_clip, 1+eps_clip) * advantages
                    loss_actor = -(torch.min(surr1, surr2)).mean()
                    loss_entropy = -c_entropy * entropies.mean()
                    loss_critic = c_critic * F.mse_loss(new_values, returns)
                    loss =  (loss_actor + loss_critic + loss_entropy) /accumulation_steps
                    loss.backward()
                    backward_count += 1
                    kl_div += torch.distributions.kl_divergence(new_dist, old_dist).mean()/accumulation_steps


                    # Step optimizer

                    if backward_count % accumulation_steps == 0 and backward_count != 0:
                        torch.nn.utils.clip_grad_norm_(model_student.parameters(), c_grad_norm)
                        optimizer.step()
                        optimizer.zero_grad()

                        # Adaptive schedule
                        for param_group in optimizer.param_groups:
                            if adaptive_schedule:
                                param_group['lr'] = tr.update_lr(param_group['lr'], kl_div)

                        # Reinitialize
                        backward_count = 0
                        kl_div = 0


                    all_losses_actor.append(loss_actor.item())
                    all_losses_critic.append(loss_critic.item())

                # Update buffers

                actions_star = model.instanciate_actions(actions, dt=cf.frames_future)
                buffer['actions_star'] += list(torch.unbind( actions_star ))


            ## Current action (numEnvs, 1)

            if(len(buffer['actions_star'])>0):
                action = buffer['actions_star'].pop(0)
                action = torch.clamp(action, min=-1, max=1)
            else:
                action = torch.dzeros(numEnvs, 1)

            if sim_count % 6 == 0 and env.sim_frame_count % sim_duration == 256:
                torchvision.io.write_video('/isaac-sim/memoryPerceiver/memory_perceiver/main/monitor/test_'  + str(sim_count) + '_vid_.mp4', 
                                            torch.stack(full_video)[:,1], 4*cf.frames)

            # Step environment

            if sim_step == 0:
                env._task.post_reset()
            env._task.pre_physics_step(action.to(task.rl_device))
            env._world.step(render=render)
            env.sim_frame_count += 1
            env._task.post_physics_step()

            sim_step+=1

        else:
            env._world.step(render=render)

    env._simulation_app.close()


if __name__ == '__main__':
    sim_duration = 260
    numEnvs = cf.batch_size
    learning_rate = 1e-4
    accumulation_steps = 8
    c_entropy = 0.01
    c_critic=0.5
    student_update_step = 1
    eps=1e-9
    c_gamma=0.99
    c_lambda=0.95
    c_grad_norm=1.0
    eps_clip=0.2
    adaptive_schedule=False
    path_model = 'weights/humanoid.pt'
    with initialize(config_path="memoryPerceiver/OmniIsaacGymEnvs/omniisaacgymenvs/cfg"):
        cfg = compose(config_name="config", overrides=["task=Humanoid", "headless=True","enable_viewport=True",
                        "+numEnvs="+str(numEnvs)])
    parse_hydra_configs(cfg)

/isaac-sim/kit/python/lib/python3.7/site-packages/hydra/_internal/defaults_list.py:251: UserWarning: In 'config': Defaults list is missing `_self_`. See https://hydra.cc/docs/upgrades/1.0_to_1.1/default_composition_order for more information
  warnings.warn(msg, UserWarning)
/isaac-sim/kit/python/lib/python3.7/site-packages/hydra/_internal/defaults_list.py:412: UserWarning: In config: Invalid overriding of hydra/job_logging:
Default list overrides requires 'override' keyword.
See https://hydra.cc/docs/next/upgrades/1.0_to_1.1/defaults_list_override for more information.

  deprecation_warning(msg)


Starting kit application with the following args:  ['/isaac-sim/exts/omni.isaac.kit/omni/isaac/kit/simulation_app.py', '/isaac-sim/apps/omni.isaac.sim.python.gym.headless.render.kit', '--/app/tokens/exe-path=/isaac-sim/kit', '--/persistent/app/viewport/displayOptions=3094', '--/rtx/materialDb/syncLoads=True', '--/rtx/hydra/materialSyncLoads=True--/omni.kit.plugin/syncUsdLoads=True', '--/app/renderer/resolution/width=1280', '--/app/renderer/resolution/height=720', '--/app/window/width=1440', '--/app/window/height=900', '--/renderer/multiGpu/enabled=True', '--/app/fastShutdown=False', '--ext-folder', '/isaac-sim/exts', '--ext-folder', '/isaac-sim/apps', '--/physics/cudaDevice=0', '--portable', '--no-window', '--allow-root']
Passing the following args to the base kit application:  ['-f', '/root/.local/share/jupyter/runtime/kernel-4a705c90-08d6-4943-80ad-cb92e521b822.json']
[Warning] [omni.isaac.kit.simulation_app] Modules: ['omni.isaac.kit.app_framework'] were loaded before SimulationApp 

2024-03-24 21:17:01 s] [Error] [carb.glinterop.plugin] GLInteropContext::init: carb::windowing is not available



|---------------------------------------------------------------------------------------------|
| Driver Version: 535.146.02    | Graphics API: Vulkan
|=============================================================================================|
| GPU | Name                             | Active | LDA | GPU Memory | Vendor-ID | LUID       |
|     |                                  |        |     |            | Device-ID | UUID       |
|---------------------------------------------------------------------------------------------|
| 0   | NVIDIA RTX A4000                 | Yes: 0 |     | 16622   MB | 10de      | 0          |
|     |                                  |        |     |            | 24b0      | 5f39c36d.. |
|---------------------------------------------------------------------------------------------|
| 1   | NVIDIA RTX A4000                 | Yes: 1 |     | 16622   MB | 10de      | 0          |
|     |                                  |        |     |            | 24b0     

: 

: 

: 

In [ ]:
!nvidia-smi

: 